# Intro to Python Packaging

This notebook focuses on a small, self-contained packaging example inside `01-intro-to-python-packaging/`:

<pre>
01-intro-to-python-packaging/
├── 01-intro-to-python-packaging.ipynb
├── pyproject.toml
└── src/
   └── example/
      ├── __init__.py
      ├── example_file_solution.py
      └── example_file.py
</pre>

A **Python file** is just a `.py` file on disk. A **module** is that file when Python imports it, which means Python gives it a name and makes its functions available to other code. A **package** is a directory of related modules that Python treats as one importable unit. In this repository, `src/example/` is the package, `example_file.py` is one module inside it, and `from src.example.example_file import add_one` is the source-layout import path.

This distinction matters because we are not only writing code, we are organizing code so Python can find and reuse it predictably. Once a file is part of a package, notebooks, tests, and installed code can all import the same function instead of duplicating logic.

This module starts with an incomplete implementation target. The first validation check is expected to fail until `src/example/example_file.py` returns the correct result.

`pyproject.toml` defines the package metadata and build backend, while `src/example/` contains the importable package code. `__init__.py` makes the directory an explicit package, `example_file.py` is the implementation target, and `example_file_solution.py` is the reference version.

Because this notebook lives in the same folder as `src/`, the source-layout import path is available directly in the notebook.


## Package Structure

```mermaid
flowchart TD
    A["src/"] --> B["example/"]
    B --> C["__init\_\_.py"]
    B --> D["example_file.py"]
    B --> E["example_file_solution.py"]
    F["Notebook or test"] --> G["from src.example.example_file import add_one"]
    G --> D
```

The important thing to notice is that Python does not import the raw folder name by accident. It follows the package path step by step: `src` -> `example` -> `example_file` -> `add_one`.

That is why package structure matters. If the folders and files are arranged clearly, the same import path can work from notebooks, tests, and installed package code.


In [1]:
# Automatically reload modules when they are edited to avoid restarting the kernel.
# Now you can edit the code in src/example/example_file.py and see the changes reflected in the notebook without restarting.
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys

for candidate in (
    Path.cwd().resolve(),
    Path.cwd().resolve() / "01-intro-to-python-packaging",
):
    if (candidate / "src").exists():
        NOTEBOOK_ROOT = candidate
        break
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

In [4]:
from src.example.example_file import add_one

Run the imported function directly in the notebook. This is the simplest possible check: can Python find the module, import the function, and call it from notebook code?

At this point the check should still fail, because the implementation target is intentionally incomplete. That failure is useful here because it confirms the import path works even before the function logic is correct.


In [ ]:
try:
    assert add_one(3) == 4
    print("add_one works correctly!")
except AssertionError:
    print("add_one does not work correctly. Please implement it in src/example/example_file.py")

## Implementation Target

Implement `add_one()` in `src/example/example_file.py`.

1. Open `src/example/example_file.py`.
2. Replace the placeholder return value with logic that increments the input by exactly `1`.
3. Re-run the notebook check above or run the command below from inside `01-intro-to-python-packaging/`.
4. Expect the validation to fail (`AssertionError`) before the implementation is complete, then pass (no output) once the function returns the correct result.


In [ ]:
# @TODO Exercise: Implement add_one in package module.
# Objective: Make sure local package imports work and the function increments by exactly 1.
# Edit files:
# - src/example/example_file.py
# Validate with:
# - ../.venv/bin/python -c "from src.example.example_file import add_one; assert add_one(3) == 4"
# Solution:
# - src/example/example_file_solution.py


This folder is also a small installable Python package example. `pyproject.toml` is the central packaging file.

**Under `[build-system]`:**

- `requires` lists the packages needed to build the distribution.
- `build-backend` selects the backend used to build the package.

**Under `[project]`:**

- `name` is the distribution name used during installation.
- `version` is the package version.
- `description` is a short summary of the package.
- `requires-python` declares the supported Python version range.

From inside `01-intro-to-python-packaging/`, install the package defined by the local `pyproject.toml` like this:

```bash
python -m pip install .
```

After installation, you can import the installed package path as `example.example_file`. The installed package still reflects the current implementation, so re-run the install command after changing `add_one()`.


In [ ]:
import sys

!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install .


In [ ]:
from example.example_file import add_one as installed_add_one

In [ ]:
print(installed_add_one(3))

## Running Checks

From inside `01-intro-to-python-packaging/`:

##### Validation command:

```zsh
../.venv/bin/python -c "from src.example.example_file import add_one; assert add_one(3) == 4"
```

**What to expect:** this command should fail until `src/example/example_file.py` is implemented

##### Reference check:

```zsh
../.venv/bin/python -c "from src.example.example_file_solution import add_one; assert add_one(3) == 4"
```

##### Installed-package check after `python -m pip install .`:

```zsh
../.venv/bin/python -c "from example.example_file import add_one; assert add_one(3) == 4"
```

**What to expect:** this command also fails until `add_one()` is implemented and the package is reinstalled

##### Optional build step if the `build` package is installed:

```zsh
python -m build
```

That command creates distribution artifacts in `dist/`.


### Quiz

1. **What is the purpose of the `__init__.py` file inside a Python package directory?**  
    [ ] To store configuration settings  
    [ ] To allow the directory to be treated as a package  
    [ ] To define project metadata like name and version  
    [ ] To compile Python code to bytecode  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** To allow the directory to be treated as a package  
      
      **Description:** The presence of **`__init__.py`** indicates package intent so modules inside can be imported consistently.
    </details>

---

2. **In the current module layout, which import path correctly refers to `example_file.py` before package installation?**  
    [ ] `example.example_file`  
    [ ] `src.example`  
    [ ] `src.example.example_file`  
    [ ] `example_file`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `src.example.example_file`  
      
      **Description:** In this source-layout module, the file lives at **`src/example/example_file.py`**, so the direct import path is **`src.example.example_file`**.
    </details>

---

3. **From this notebook folder, what command installs the package defined by the local `pyproject.toml`?**  
    [ ] `pip install pyproject.toml`  
    [ ] `python setup.py install`  
    [ ] `python -m pip install .`  
    [ ] `pip build .`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `python -m pip install .`  
      
      **Description:** `.` refers to the current directory, which contains the module-specific **`pyproject.toml`**.
    </details>

---

4. **Which table in `pyproject.toml` specifies the build backend and build requirements?**  
    [ ] `[project]`  
    [ ] `[tool.pytest]`  
    [ ] `[build-system]`  
    [ ] `[metadata]`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `[build-system]`  
      
      **Description:** The **`[build-system]`** table declares both the build backend and the packages required to build the distribution.


---

5. **Which key in the `[project]` section sets the public distribution name of the package?**  
    [ ] `package_name`  
    [ ] `distribution`  
    [ ] `name`  
    [ ] `project_title`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `name`  
      
      **Description:** The **`name`** field defines the installable distribution name.
    </details>

---

6. **Which key in `[project]` specifies the minimum Python version the package supports?**  
    [ ] `python-version`  
    [ ] `requires-python`  
    [ ] `python`  
    [ ] `min-version`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `requires-python`  
      
      **Description:** The **`requires-python`** field tells installers which Python versions are compatible.
    </details>

---

7. **After installing with `python -m pip install .`, which import is correct for this package?**  
    [ ] `from src.example.example_file import add_one`  
    [ ] `from example.example_file import add_one`  
    [ ] `import src.example_file`  
    [ ] `import example_file`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `from example.example_file import add_one`  
      
      **Description:** Once installed, you import from the installed package namespace (`example`).
    </details>

---

8. **Which command manually adds a directory to Python's module search path?**  
    [ ] `os.path.insert()`  
    [ ] `sys.path.append()`  
    [ ] `pythonpath.add()`  
    [ ] `sys.add_path()`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `sys.path.append()`  
      
      **Description:** `sys.path.append('../')` adds a directory to `sys.path` for import resolution.
    </details>
